## 1. Environment Setup

In [1]:
import os
import sys
import random
import shutil
import numpy as np
import cv2
import torch
from tqdm import tqdm

# Clone DepthAnythingV2
!git clone https://github.com/DepthAnything/Depth-Anything-V2 > /dev/null 2>&1
%cd /kaggle/working/Depth-Anything-V2
!pip install -r requirements.txt -q

# Clone LLFormer
%cd /kaggle/working
!git clone https://github.com/TaoWangzj/LLFormer.git > /dev/null 2>&1
os.chdir('/kaggle/working/LLFormer')
!cd pytorch-gradual-warmup-lr && python setup.py install > /dev/null 2>&1
!pip install natsort yacs gdown scikit-image h5py joblib -q
print("Done!")

/kaggle/working/Depth-Anything-V2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 2.5 MB/s eta 0:00:00 0:00:01
/kaggle/working
Done!


## 2. Data Preparation

In [2]:
!ls /kaggle/input/datasets/mavislei/exdark-360/sampled_originals

Bicycle  Boat  Bottle  Bus  Car  Cat  Chair  Cup  Dog  Motorbike  People  Table


In [3]:
DATASET_ROOT = "/kaggle/input/datasets/mavislei/exdark-360"
IMG_ROOT = os.path.join(DATASET_ROOT, "sampled_originals")

image_paths = []
for category in sorted(os.listdir(IMG_ROOT)):
    cat_dir = os.path.join(IMG_ROOT, category)
    if os.path.isdir(cat_dir):
        for fname in sorted(os.listdir(cat_dir)):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                image_paths.append(os.path.join(cat_dir, fname))

print(f"Found {len(image_paths)} images across {len(os.listdir(IMG_ROOT))} categories")

# Output structure: one folder per image, containing rgb/ and depth/ subfolders,
# each holding the 5 pipeline versions
PIPELINES = ["dark", "clahe", "gamma", "msr", "llformer"]
OUT_ROOT = "/kaggle/working/outputs"

for img_path in image_paths:
    img_stem = os.path.splitext(os.path.basename(img_path))[0]
    img_out_dir = os.path.join(OUT_ROOT, img_stem)
    os.makedirs(os.path.join(img_out_dir, "rgb"), exist_ok=True)
    os.makedirs(os.path.join(img_out_dir, "depth"), exist_ok=True)

print(f"Output directories ready for {len(image_paths)} images under:", OUT_ROOT)

Found 360 images across 12 categories
Output directories ready for 360 images under: /kaggle/working/outputs


## 3. Load DepthAnythingV2

In [4]:
import torch
import sys

sys.path.append('/kaggle/working/Depth-Anything-V2')
from depth_anything_v2.dpt import DepthAnythingV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

depth_model_path = '/kaggle/input/models/artemmmtry/depth-anything-v2/pytorch/small-model/1/depth_anything_v2_vits.pth'
depth_model = DepthAnythingV2(encoder='vits', features=64, out_channels=[48, 96, 192, 384])
depth_model.load_state_dict(torch.load(depth_model_path, map_location=device))
depth_model = depth_model.to(device).eval()
print("DepthAnythingV2 loaded!")

xFormers not available
xFormers not available


Using device: cuda
DepthAnythingV2 loaded!


## 4. Download LLFormer Weights (LOL)

In [5]:
%cd /kaggle/working/LLFormer
!mkdir -p checkpoints/LOL
!gdown --folder https://drive.google.com/drive/folders/1J7NvvPsCtT0j8Rd9ombJ6sVIC6v0Xweb -O ./checkpoints/LOL/ > /dev/null 2>&1

# Verify the download actually worked
downloaded_files = os.listdir('./checkpoints/LOL/')
assert len(downloaded_files) > 0, "Download failed — checkpoints/LOL/ is empty!"
print(f"LLFormer LOL weights downloaded! Files: {downloaded_files}")

/kaggle/working
LLFormer LOL weights downloaded! Files: ['LOL']


## 5. Step A: Batch Enhancement — Traditional Methods (Dark / CLAHE / Gamma / MSR)

Runs the four non-learning-based pipelines across all ExDark images and saves the enhanced RGB outputs. LLFormer is handled separately in a later step due to model-specific input constraints (see Step B).

In [6]:
import shutil
shutil.rmtree('/kaggle/working/outputs', ignore_errors=True)
print("Cleared old outputs folder")

Cleared old outputs folder


In [7]:
# ---- Enhancement functions ----

def enhance_dark(img):
    """No enhancement — the original low-light image."""
    return img.copy()


def enhance_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8)):
    """Apply CLAHE to the L channel in LAB colour space."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid_size
    )
    l_enhanced = clahe.apply(l_channel)

    lab_enhanced = cv2.merge([
        l_enhanced,
        a_channel,
        b_channel
    ])

    return cv2.cvtColor(
        lab_enhanced,
        cv2.COLOR_LAB2BGR
    )


def enhance_gamma(img, gamma=2.2):
    """Gamma correction using an inverse-gamma lookup table."""
    inv_gamma = 1.0 / gamma

    table = np.array([
        ((i / 255.0) ** inv_gamma) * 255.0
        for i in range(256)
    ]).astype(np.uint8)

    return cv2.LUT(img, table)


def enhance_msr(img, sigmas=(15, 80, 250)):
    """
    Multi-Scale Retinex with per-channel Min-Max normalization.
    """
    img_f = img.astype(np.float32) + 1.0
    msr_log = np.zeros_like(img_f, dtype=np.float32)

    for sigma in sigmas:
        blurred = cv2.GaussianBlur(
            img_f,
            (0, 0),
            sigma
        )

        msr_log += (
            np.log(img_f)
            - np.log(blurred + 1e-6)
        )

    msr_log /= len(sigmas)

    output = np.zeros_like(
        msr_log,
        dtype=np.float32
    )

    for channel_idx in range(3):
        channel = msr_log[:, :, channel_idx]

        channel_min = channel.min()
        channel_max = channel.max()

        if channel_max - channel_min < 1e-8:
            output[:, :, channel_idx] = 0
            continue

        output[:, :, channel_idx] = (
            (channel - channel_min)
            / (channel_max - channel_min)
            * 255.0
        )

    return np.clip(
        output,
        0,
        255
    ).astype(np.uint8)

In [8]:
IMG_ROOT = "/kaggle/input/datasets/mavislei/exdark-360/sampled_originals"

image_paths = []
image_categories = {}  # img_stem -> category

for category in sorted(os.listdir(IMG_ROOT)):
    cat_dir = os.path.join(IMG_ROOT, category)
    if os.path.isdir(cat_dir):
        for fname in sorted(os.listdir(cat_dir)):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                full_path = os.path.join(cat_dir, fname)
                image_paths.append(full_path)
                img_stem = os.path.splitext(fname)[0]
                image_categories[img_stem] = category

print(f"Found {len(image_paths)} images across {len(image_categories)} stems, "
      f"{len(set(image_categories.values()))} categories")

Found 360 images across 360 stems, 12 categories


In [9]:
OUT_ROOT = "/kaggle/working/outputs"

for img_path in image_paths:
    img_stem = os.path.splitext(os.path.basename(img_path))[0]
    category = image_categories[img_stem]
    img_out_dir = os.path.join(OUT_ROOT, category, img_stem)
    os.makedirs(os.path.join(img_out_dir, "rgb"), exist_ok=True)
    os.makedirs(os.path.join(img_out_dir, "depth"), exist_ok=True)

print(f"Output directories ready under: {OUT_ROOT}")

Output directories ready under: /kaggle/working/outputs


In [10]:
# ---- Step A: Dark / CLAHE / Gamma / MSR — batch enhancement (RGB only, no depth yet) ----
TRADITIONAL_ENHANCERS = {
    "dark": enhance_dark,
    "clahe": enhance_clahe,
    "gamma": enhance_gamma,
    "msr": enhance_msr,
}

for img_path in tqdm(image_paths):
    img_stem = os.path.splitext(os.path.basename(img_path))[0]
    img_out_dir = os.path.join(OUT_ROOT, image_categories[img_stem], img_stem)
    image = cv2.imread(img_path)
    for pipeline_name, enhance_fn in TRADITIONAL_ENHANCERS.items():
        enhanced = enhance_fn(image)
        rgb_path = os.path.join(img_out_dir, "rgb", f"{pipeline_name}.png")
        cv2.imwrite(rgb_path, enhanced)

print(f"Step A done: {len(image_paths)} images \u00d7 4 traditional pipelines saved.")

100%|██████████| 360/360 [20:19<00:00,  3.39s/it]  

Step A done: 360 images × 4 traditional pipelines saved.


## 5. Step B: LLFormer — Batch Enhancement (Official Script)

Exports the original low-light images as PNG (required by LLFormer's official `test.py`), runs the pretrained LOL model via the official script, then moves the results into the per-image/per-category output structure alongside the traditional methods.

In [11]:
shutil.rmtree('/kaggle/working/llformer_enhanced', ignore_errors=True)

In [12]:
# ---- Step B.1: Export images as PNG and resize to a maximum dimension of 512 px ----

LLFORMER_INPUT_DIR = "/kaggle/working/llformer_input_png"
os.makedirs(LLFORMER_INPUT_DIR, exist_ok=True)

MAX_SIZE = 512

for img_path in tqdm(image_paths):
    img_stem = os.path.splitext(os.path.basename(img_path))[0]
    image = cv2.imread(img_path)
    h, w = image.shape[:2]
    scale = MAX_SIZE / max(h, w)
    if scale < 1:  # only downscale, never upscale
        image = cv2.resize(image, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    cv2.imwrite(os.path.join(LLFORMER_INPUT_DIR, f"{img_stem}.png"), image)

print(f"Exported {len(image_paths)} images (max {MAX_SIZE}px) to {LLFORMER_INPUT_DIR}")

100%|██████████| 360/360 [00:09<00:00, 39.78it/s]

Exported 360 images (max 512px) to /kaggle/working/llformer_input_png


In [13]:
# ---- Step B.2: Run LLFormer's official test.py (handles its own padding — mul=16) ----

os.chdir('/kaggle/working/LLFormer')

!python test.py \
    --input_dir /kaggle/working/llformer_input_png/ \
    --result_dir /kaggle/working/llformer_enhanced/ \
    --weights ./checkpoints/LOL/LOL/models/model_bestPSNR.pth


enhanced_files = os.listdir('/kaggle/working/llformer_enhanced')
print(f"LLFormer produced {len(enhanced_files)} output files (expected {len(image_paths)})")

restoring images......
1/360
2/360
3/360
4/360
5/360
6/360
7/360
8/360
9/360
10/360
11/360
12/360
13/360
14/360
15/360
16/360
17/360
18/360
19/360
20/360
21/360
22/360
23/360
24/360
25/360
26/360
27/360
28/360
29/360
30/360
31/360
32/360
33/360
34/360
35/360
36/360
37/360
38/360
39/360
40/360
41/360
42/360
43/360
44/360
45/360
46/360
47/360
48/360
49/360
50/360
51/360
52/360
53/360
54/360
55/360
56/360
57/360
58/360
59/360
60/360
61/360
62/360
63/360
64/360
65/360
66/360
67/360
68/360
69/360
70/360
71/360
72/360
73/360
74/360
75/360
76/360
77/360
78/360
79/360
80/360
81/360
82/360
83/360
84/360
85/360
86/360
87/360
88/360
89/360
90/360
91/360
92/360
93/360
94/360
95/360
96/360
97/360
98/360
99/360
100/360
101/360
102/360
103/360
104/360
105/360
106/360
107/360
108/360
109/360
110/360
111/360
112/360
113/360
114/360
115/360
116/360
117/360
118/360
119/360
120/360
121/360
122/360
123/360
124/360
125/360
126/360
127/360
128/360
129/360
130/360
131/360
132/360
133/360
134/360
135/360
136/3

In [14]:
# ---- Step B.3: Move LLFormer outputs into the outputs/{category}/{img_stem}/rgb/ structure ----

LLFORMER_OUT_DIR = "/kaggle/working/llformer_enhanced"

missing = []
for img_path in tqdm(image_paths):
    img_stem = os.path.splitext(os.path.basename(img_path))[0]
    category = image_categories[img_stem]

    src = os.path.join(LLFORMER_OUT_DIR, f"{img_stem}.png")
    dst = os.path.join(OUT_ROOT, category, img_stem, "rgb", "llformer.png")

    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        missing.append(img_stem)

print(f"Moved {len(image_paths) - len(missing)} LLFormer outputs into place.")
if missing:
    print(f"WARNING: {len(missing)} images missing LLFormer output, e.g.: {missing[:5]}")

100%|██████████| 360/360 [00:00<00:00, 3010.82it/s]

Moved 360 LLFormer outputs into place.


## Step C: Depth Inference (All 5 Pipelines)

Runs the relative-depth model (DepthAnythingV2 vits) on all 5 enhanced RGB versions of each image. Saves both the raw depth array (`.npy`, needed later for edge-alignment scoring) and a colorized visualization (`.png`, for the PPT).

In [15]:
def depth_to_vis(depth):
    """
    Convert a raw depth map into a colour visualisation.
    This is only for display; the raw .npy depth is used for evaluation.
    """
    depth = np.asarray(depth, dtype=np.float32)

    valid = np.isfinite(depth)

    if valid.sum() == 0:
        return np.zeros(
            (*depth.shape, 3),
            dtype=np.uint8
        )

    depth_valid = depth[valid]

    depth_min = depth_valid.min()
    depth_max = depth_valid.max()

    depth_norm = np.zeros_like(
        depth,
        dtype=np.float32
    )

    depth_norm[valid] = (
        depth[valid] - depth_min
    ) / (
        depth_max - depth_min + 1e-8
    )

    depth_uint8 = (
        depth_norm * 255
    ).astype(np.uint8)

    return cv2.applyColorMap(
        depth_uint8,
        cv2.COLORMAP_TURBO
    )

In [16]:
# ---- Step C: Depth inference across all 5 pipelines ----

PIPELINES = ["dark", "clahe", "gamma", "msr", "llformer"]

for img_path in tqdm(image_paths):
    img_stem = os.path.splitext(os.path.basename(img_path))[0]
    category = image_categories[img_stem]
    img_out_dir = os.path.join(OUT_ROOT, category, img_stem)

    for pipeline_name in PIPELINES:
        rgb_path = os.path.join(img_out_dir, "rgb", f"{pipeline_name}.png")
        enhanced = cv2.imread(rgb_path)
        enhanced_rgb = cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)

        depth = depth_model.infer_image(enhanced_rgb)

        # Save raw depth array (needed for Step D's edge-alignment computation)
        np.save(os.path.join(img_out_dir, "depth", f"{pipeline_name}.npy"), depth)
        # Save a colourised depth map for qualitative visualisation
        cv2.imwrite(os.path.join(img_out_dir, "depth", f"{pipeline_name}.png"), depth_to_vis(depth))

print(f"Step C done: depth inferred for {len(image_paths)} images \u00d7 {len(PIPELINES)} pipelines.")

100%|██████████| 360/360 [04:48<00:00,  1.25it/s]

Step C done: depth inferred for 360 images × 5 pipelines.


## Step D: Edge-Alignment Scoring (Canny vs. Sobel → Precision / Recall / F1)

Since ExDark has no ground-truth depth, structural consistency is used as a proxy: Canny edges from the RGB image are compared against Sobel-detected discontinuities in the depth map. A small tolerance (2px dilation) allows near-miss alignment, following standard boundary-matching practice. Results are averaged per pipeline across all 360 images.

In [17]:
# ---- Step D: Canny (RGB) vs. Sobel (depth) edge-alignment
#      Precision / Recall / F1 ----

import os
import csv
import cv2
import numpy as np
from tqdm import tqdm


def compute_edge_metrics(rgb_img, depth_arr, tolerance_px=2):
    """
    Compute neighbourhood-based edge-alignment metrics between an RGB image
    and its predicted depth map.

    Precision:
        Proportion of predicted depth-edge pixels that lie within the
        tolerance neighbourhood of an image edge.

    Recall:
        Proportion of image-edge pixels that lie within the tolerance
        neighbourhood of a predicted depth edge.

    F1:
        Harmonic mean of Precision and Recall.
    """

    if rgb_img is None:
        raise ValueError("The RGB image could not be loaded.")

    if depth_arr is None or depth_arr.size == 0:
        raise ValueError("The depth array is empty or invalid.")

    # ---------------------------------------------------------
    # 1. Extract image edges using Canny
    # ---------------------------------------------------------
    gray = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2GRAY)
    canny_edges = cv2.Canny(gray, 100, 200) > 0

    # ---------------------------------------------------------
    # 2. Extract depth edges using Sobel
    # ---------------------------------------------------------
    depth_norm = cv2.normalize(
        depth_arr,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    ).astype(np.uint8)

    sobel_x = cv2.Sobel(
        depth_norm,
        cv2.CV_64F,
        1,
        0,
        ksize=3
    )

    sobel_y = cv2.Sobel(
        depth_norm,
        cv2.CV_64F,
        0,
        1,
        ksize=3
    )

    sobel_magnitude = np.sqrt(
        sobel_x ** 2 + sobel_y ** 2
    )

    # Retain the strongest 10% of Sobel gradient responses
    sobel_threshold = np.percentile(
        sobel_magnitude,
        90
    )

    depth_edges = sobel_magnitude > sobel_threshold

    # ---------------------------------------------------------
    # 3. Create tolerance neighbourhoods using dilation
    # ---------------------------------------------------------
    kernel_size = tolerance_px * 2 + 1

    kernel = np.ones(
        (kernel_size, kernel_size),
        dtype=np.uint8
    )

    canny_dilated = cv2.dilate(
        canny_edges.astype(np.uint8),
        kernel
    ) > 0

    depth_dilated = cv2.dilate(
        depth_edges.astype(np.uint8),
        kernel
    ) > 0

    # ---------------------------------------------------------
    # 4. Precision
    # ---------------------------------------------------------
    # Count predicted depth-edge pixels that lie within the
    # tolerance neighbourhood of an image edge.
    matched_depth_edges = np.sum(
        depth_edges & canny_dilated
    )

    total_depth_edges = np.sum(depth_edges)

    if total_depth_edges > 0:
        precision = (
            matched_depth_edges / total_depth_edges
        )
    else:
        precision = 0.0

    # ---------------------------------------------------------
    # 5. Recall
    # ---------------------------------------------------------
    # Count image-edge pixels that lie within the tolerance
    # neighbourhood of a predicted depth edge.
    matched_image_edges = np.sum(
        canny_edges & depth_dilated
    )

    total_image_edges = np.sum(canny_edges)

    if total_image_edges > 0:
        recall = (
            matched_image_edges / total_image_edges
        )
    else:
        recall = 0.0

    # ---------------------------------------------------------
    # 6. F1-score
    # ---------------------------------------------------------
    if precision + recall > 0:
        f1 = (
            2 * precision * recall
            / (precision + recall)
        )
    else:
        f1 = 0.0

    return precision, recall, f1


# -------------------------------------------------------------
# Evaluate all images and all five pipelines
# -------------------------------------------------------------
edge_results = {
    pipeline_name: {
        "precision": [],
        "recall": [],
        "f1": []
    }
    for pipeline_name in PIPELINES
}

per_image_rows = []

for img_path in tqdm(
    image_paths,
    desc="Computing edge-alignment metrics"
):
    img_stem = os.path.splitext(
        os.path.basename(img_path)
    )[0]

    category = image_categories[img_stem]

    img_out_dir = os.path.join(
        OUT_ROOT,
        category,
        img_stem
    )

    row = {
        "image_stem": img_stem,
        "category": category
    }

    for pipeline_name in PIPELINES:
        rgb_path = os.path.join(
            img_out_dir,
            "rgb",
            f"{pipeline_name}.png"
        )

        depth_path = os.path.join(
            img_out_dir,
            "depth",
            f"{pipeline_name}.npy"
        )

        if not os.path.exists(rgb_path):
            raise FileNotFoundError(
                f"Missing RGB image: {rgb_path}"
            )

        if not os.path.exists(depth_path):
            raise FileNotFoundError(
                f"Missing depth array: {depth_path}"
            )

        rgb = cv2.imread(rgb_path)
        depth = np.load(depth_path)

        precision, recall, f1 = compute_edge_metrics(
            rgb,
            depth,
            tolerance_px=2
        )

        edge_results[pipeline_name]["precision"].append(
            precision
        )

        edge_results[pipeline_name]["recall"].append(
            recall
        )

        edge_results[pipeline_name]["f1"].append(
            f1
        )

        row[f"{pipeline_name}_precision"] = precision
        row[f"{pipeline_name}_recall"] = recall
        row[f"{pipeline_name}_f1"] = f1

    per_image_rows.append(row)


# -------------------------------------------------------------
# Print mean results for each pipeline
# -------------------------------------------------------------
print(
    "\nMean edge-alignment metrics by pipeline "
    "(higher values indicate stronger alignment):"
)

summary_rows = []

for pipeline_name in PIPELINES:
    precision_scores = edge_results[
        pipeline_name
    ]["precision"]

    recall_scores = edge_results[
        pipeline_name
    ]["recall"]

    f1_scores = edge_results[
        pipeline_name
    ]["f1"]

    mean_precision = np.mean(precision_scores)
    mean_recall = np.mean(recall_scores)
    mean_f1 = np.mean(f1_scores)

    std_precision = np.std(precision_scores)
    std_recall = np.std(recall_scores)
    std_f1 = np.std(f1_scores)

    n_images = len(f1_scores)

    print(
        f"{pipeline_name:10s} | "
        f"Precision: {mean_precision:.4f} | "
        f"Recall: {mean_recall:.4f} | "
        f"F1: {mean_f1:.4f} | "
        f"n = {n_images}"
    )

    summary_rows.append({
        "pipeline": pipeline_name,
        "mean_precision": mean_precision,
        "std_precision": std_precision,
        "mean_recall": mean_recall,
        "std_recall": std_recall,
        "mean_f1": mean_f1,
        "std_f1": std_f1,
        "n_images": n_images
    })


# -------------------------------------------------------------
# Save per-image metrics
# -------------------------------------------------------------
per_image_csv_path = (
    "/kaggle/working/"
    "exdark_edge_alignment_per_image.csv"
)

per_image_fieldnames = [
    "image_stem",
    "category"
]

for pipeline_name in PIPELINES:
    per_image_fieldnames.extend([
        f"{pipeline_name}_precision",
        f"{pipeline_name}_recall",
        f"{pipeline_name}_f1"
    ])

with open(
    per_image_csv_path,
    "w",
    newline="",
    encoding="utf-8"
) as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=per_image_fieldnames
    )

    writer.writeheader()
    writer.writerows(per_image_rows)


# -------------------------------------------------------------
# Save summary metrics
# -------------------------------------------------------------
summary_csv_path = (
    "/kaggle/working/"
    "exdark_edge_alignment_summary.csv"
)

summary_fieldnames = [
    "pipeline",
    "mean_precision",
    "std_precision",
    "mean_recall",
    "std_recall",
    "mean_f1",
    "std_f1",
    "n_images"
]

with open(
    summary_csv_path,
    "w",
    newline="",
    encoding="utf-8"
) as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=summary_fieldnames
    )

    writer.writeheader()
    writer.writerows(summary_rows)


print(
    "\nSaved per-image metrics to:"
)
print(per_image_csv_path)

print(
    "\nSaved summary metrics to:"
)
print(summary_csv_path)

Computing edge-alignment metrics: 100%|██████████| 360/360 [01:16<00:00,  4.71it/s]


Mean edge-alignment metrics by pipeline (higher values indicate stronger alignment):
dark       | Precision: 0.3547 | Recall: 0.4294 | F1: 0.3181 | n = 360
clahe      | Precision: 0.5029 | Recall: 0.3815 | F1: 0.3678 | n = 360
gamma      | Precision: 0.5087 | Recall: 0.4135 | F1: 0.3839 | n = 360
msr        | Precision: 0.5879 | Recall: 0.3599 | F1: 0.3928 | n = 360
llformer   | Precision: 0.5883 | Recall: 0.4377 | F1: 0.4612 | n = 360

Saved per-image metrics to:
/kaggle/working/exdark_edge_alignment_per_image.csv

Saved summary metrics to:
/kaggle/working/exdark_edge_alignment_summary.csv


In [18]:
# Sanity check: raw edge pixel counts per pipeline (averaged)
edge_counts = {p: [] for p in PIPELINES}
for img_path in tqdm(image_paths[:30]):  
    img_stem = os.path.splitext(os.path.basename(img_path))[0]
    category = image_categories[img_stem]
    img_out_dir = os.path.join(OUT_ROOT, category, img_stem)
    for pipeline_name in PIPELINES:
        rgb = cv2.imread(os.path.join(img_out_dir, "rgb", f"{pipeline_name}.png"))
        gray = cv2.cvtColor(rgb, cv2.COLOR_BGR2GRAY)
        canny_edges = cv2.Canny(gray, 100, 200)
        edge_counts[pipeline_name].append(np.sum(canny_edges > 0))

print("Mean Canny edge pixel count by pipeline:")
for p in PIPELINES:
    print(f"  {p:10s}: {np.mean(edge_counts[p]):.0f}")

100%|██████████| 30/30 [00:03<00:00,  8.84it/s]

Mean Canny edge pixel count by pipeline:
  dark      : 23140
  clahe     : 53935
  gamma     : 36826
  msr       : 42487
  llformer  : 17533


In [19]:
# Sanity check: depth-edge pixel counts per pipeline (averaged)
depth_edge_counts = {p: [] for p in PIPELINES}

for img_path in tqdm(image_paths[:30]): # Sample the first 30 images to check the trend
    img_stem = os.path.splitext(os.path.basename(img_path))[0]
    category = image_categories[img_stem]
    img_out_dir = os.path.join(OUT_ROOT, category, img_stem)

    for pipeline_name in PIPELINES:
        depth = np.load(os.path.join(img_out_dir, "depth", f"{pipeline_name}.npy"))
        depth_norm = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        sobel_x = cv2.Sobel(depth_norm, cv2.CV_64F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(depth_norm, cv2.CV_64F, 0, 1, ksize=3)
        sobel_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)
        thresh = np.percentile(sobel_mag, 90)  
        depth_edges = sobel_mag > thresh
        depth_edge_counts[pipeline_name].append(np.sum(depth_edges))

print("Mean depth-edge pixel count by pipeline:")
for p in PIPELINES:
    print(f"  {p:10s}: {np.mean(depth_edge_counts[p]):.0f}")

100%|██████████| 30/30 [00:02<00:00, 10.08it/s]

Mean depth-edge pixel count by pipeline:
  dark      : 51962
  clahe     : 52732
  gamma     : 54664
  msr       : 52274
  llformer  : 15893


## Optional: Package Experimental Results for Download

This step collects the generated outputs and summary files into a single archive for convenient download. It is not required to reproduce the evaluation results.

In [20]:
import shutil
import os

os.makedirs("/kaggle/working/exdark_results", exist_ok=True)

shutil.copy(
    "/kaggle/working/exdark_edge_alignment_summary.csv",
    "/kaggle/working/exdark_results/"
)

shutil.copy(
    "/kaggle/working/exdark_edge_alignment_per_image.csv",
    "/kaggle/working/exdark_results/"
)

if os.path.exists("/kaggle/working/outputs"):
    shutil.copytree(
        "/kaggle/working/outputs",
        "/kaggle/working/exdark_results/outputs",
        dirs_exist_ok=True
    )

shutil.make_archive(
    "/kaggle/working/exdark_results",
    "zip",
    "/kaggle/working/exdark_results"
)

print("Done!")
print("/kaggle/working/exdark_results.zip")

Done!
/kaggle/working/exdark_results.zip
